# Silver layer — generic document structure extraction

This Silver layer is designed to be generic for Dutch and English PDFs/reports/theses/papers.

Main goals:
- Avoid using table-of-contents entries as `main_text`.
- Detect the actual body start by checking whether a heading is followed by real paragraph text.
- Separate title/front matter, table of contents, main text, references and appendices.
- Build sections and chunks from actual document body text, not from TOC rows.
- Run fully locally and independently of Ollama.

No document-specific terms, names, organizations, locations or project topics are hardcoded.


In [8]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime
from collections import Counter


Input: ..\..\Data\bronze\doc_01.txt
Raw characters: 25204


# Configuration

In [ ]:
DOCUMENT_ID = "doc_01"
BASE_DIR = Path("../../Data")
BRONZE_DIR = BASE_DIR / "bronze"
SILVER_DIR = BASE_DIR / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHUNK_WORDS = 900
MAX_CHUNK_WORDS = 1200
MIN_CHUNK_WORDS = 120
CHUNK_OVERLAP_WORDS = 180

# Generic language cues only.
DUTCH_CUES = {"hoofdstuk", "inhoudsopgave", "samenvatting", "voorwoord", "bijlage", "bijlagen", "bibliografie", "onderzoek", "methode", "resultaten", "conclusie"}
ENGLISH_CUES = {"chapter", "contents", "table of contents", "abstract", "preface", "appendix", "appendices", "references", "introduction", "methodology", "results", "conclusion"}

MONTHS = (
    "january|february|march|april|may|june|july|august|september|october|november|december|"
    "januari|februari|maart|april|mei|juni|juli|augustus|oktober|november|december"
)

In [ ]:
def read_text_file(path: Path) -> str:
    """
    Read text from a file, ensuring UTF-8 encoding and replacing errors.
    """
    return path.read_text(encoding="utf-8", errors="replace")

def write_json(obj, path: Path):
    """
    Write a Python object to a JSON file with UTF-8 encoding.
    """
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def find_input_file():
    """
    Search for the input text file in the bronze directory using multiple patterns.
    """
    patterns = [f"{DOCUMENT_ID}.txt", f"{DOCUMENT_ID}_raw.txt", "*.txt"]
    for pat in patterns:
        matches = sorted(BRONZE_DIR.glob(pat))
        if matches:
            return matches[0]
    # Fallback when notebook is tested next to exported text files.
    local_patterns = [f"{DOCUMENT_ID}.txt", f"{DOCUMENT_ID}_raw.txt", "*.txt"]
    for pat in local_patterns:
        matches = sorted(Path.cwd().glob(pat))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No bronze text file found in {BRONZE_DIR}")

input_path = find_input_file()
raw_text = read_text_file(input_path)
print("Input:", input_path)
print("Raw characters:", len(raw_text))


# Cleaning 

In [ ]:
def normalize_unicode(text: str) -> str:
    """"
    Normalize Unicode characters to a consistent form and replace common problematic characters.
    """
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00ad": "",     # soft hyphen
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2026": "...",
        "\xa0": " ",
    }
    for a, b in replacements.items():
        text = text.replace(a, b)
    return text


def fix_pdf_spacing(text: str) -> str:
    """
    Repair common PDF extraction issues without removing meaningful structure.
    """
    # Repair common PDF extraction issues without removing meaningful structure.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)   # hyphenated line break
    text = re.sub(r"[ \t]+", " ", text)
    # Fix common stuck words from PDF extraction, generic not document-specific.
    text = re.sub(r"\bto(?=the\b)", "to ", text, flags=re.I)
    text = re.sub(r"\binto(?=the\b)", "into ", text, flags=re.I)
    text = re.sub(r"\bfrom(?=raw\b)", "from ", text, flags=re.I)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_repeated_page_noise(text: str) -> str:
    """
    Remove common repeated page markers and isolated page numbers that are not meaningful content.
    """
    lines = [re.sub(r"\s+", " ", x).strip() for x in text.splitlines()]
    # Remove very common page markers and isolated page numbers.
    cleaned = []
    for line in lines:
        low = line.lower()
        if re.fullmatch(r"\d+\s*[|/\\]\s*(p\s*a\s*g\s*e|page)", low):
            continue
        if re.fullmatch(r"(page|pagina)\s+\d+", low):
            continue
        cleaned.append(line)
    return "\n".join(cleaned)


def clean_text(text: str) -> str:
    """
    Apply a series of cleaning steps to the extracted text
    to improve quality for downstream processing.
    """
    text = normalize_unicode(text)
    text = fix_pdf_spacing(text)
    text = clean_repeated_page_noise(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

cleaned_text = clean_text(raw_text)
print("Cleaned characters:", len(cleaned_text))
print(cleaned_text[:1200])


Cleaned characters: 24263
Climate Change Risk and Impact Assessment for Kent and Medway Part 2: Industry 1

Climate Change Risk
and Impact Assessment
for Kent and Medway

Part 2:
Industry Sector Summary

November 2019

Climate Change Risk and Impact Assessment for Kent and Medway Part 2: Industry 2
I. Industry
I.1 Key Characteristics

As with the United Kingdom, Kent's economy is predominantly driven by small and
medium enterprises (SMEs). Approximately 90% of companies in Kent and Medway
are micro-businesses (less than 10 employees), while the majority of the other 10%
are small businesses with less than 50 employees. Medium and large businesses
make up less than 2% of all businesses within Kent and Medway.

In comparison with the UK, the economy of Kent and Medway comprises more micro
businesses and fewer small, medium and large businesses. Large businesses
account for only 0.32% of all businesses in Kent and Medway, compared to 0.38% of
all businesses nationwide. Data on the number 

# Generic line config

In [ ]:
def clean_line(line):
    """
    Clean a single line of text for analysis, removing extra whitespace.
    """
    return re.sub(r"\s+", " ", str(line or "")).strip()

def get_lines(text):
    """
    Split text into cleaned lines for analysis.
    """
    return [clean_line(x) for x in str(text or "").splitlines()]

lines = get_lines(cleaned_text)


def detect_language(text: str) -> str:
    """
    Detect the language of the text based on the presence of common Dutch and English cues.
    """
    sample = str(text or "")[:20000].lower()
    nl = sum(sample.count(x) for x in DUTCH_CUES)
    en = sum(sample.count(x) for x in ENGLISH_CUES)
    if nl > en:
        return "nl"
    if en > nl:
        return "en"
    return "unknown"


def is_contents_heading(line):
    """
    Check if a line is likely a contents/TOC heading based on common keywords.
    """
    low = clean_line(line).lower().strip(":")
    return low in {"contents", "table of contents", "inhoudsopgave", "inhoud", "toc"}


def has_dot_leader(line):
    """
    Check if a line has a dot leader (e.g. "1 Introduction .......... 5").
    """
    return bool(re.search(r"\.{4,}", line))


def has_trailing_page_number(line):
    """
    Check if a line ends with a page number, which is common in TOC entries.
    """
    return bool(re.search(r"\s+\d{1,4}\s*$", line))


def is_numbered_heading(line):
    """
    Check if a line looks like a numbered heading, e.g. "1 Introduction" or "Chapter 2.3 Method".
    """
    line = clean_line(line)
    if len(line) > 160:
        return False
    return bool(re.match(r"^(?:chapter|hoofdstuk\s+)?\d+(?:\.\d+){0,6}\s+\S+", line, flags=re.I))


def strip_heading_page_number(line):
    """
    If a line looks like a heading/TOC row with a trailing page number, remove the page number.
    """
    line = clean_line(line)
    # Remove TOC-style dot leaders and final page number.
    line = re.sub(r"\.{4,}\s*\d{1,4}\s*$", "", line).strip()
    # Remove final page number when a line looks like a heading/TOC row, not normal prose.
    if is_numbered_heading(line) and re.search(r"\s+\d{1,4}\s*$", line):
        core = re.sub(r"\s+\d{1,4}\s*$", "", line).strip()
        # Only remove if the remaining line is short enough and not a sentence.
        if len(core.split()) <= 12 and not core.endswith((".", "?", "!")):
            return core
    return line


def is_toc_entry(line, in_toc_region=False):
    """
    Heuristically determine if a line is likely a Table of Contents entry based on various cues.
    """
    line = clean_line(line)
    if not line:
        return False
    low = line.lower()
    if is_contents_heading(line):
        return True
    if has_dot_leader(line) and has_trailing_page_number(line):
        return True
    # In a detected TOC region, lines like "1 Introduction 1" are TOC rows.
    if in_toc_region and is_numbered_heading(line) and has_trailing_page_number(line):
        stripped = strip_heading_page_number(line)
        if stripped != line:
            return True
    # Compact appendix TOC rows, e.g. "Appendix 36 A Supplementary Material 36 ..."
    if in_toc_region and re.match(r"^(appendix|appendices|bijlage|bijlagen)\b", low) and re.search(r"\b\d{1,4}\b", low):
        return True
    return False


def is_reference_heading(line):
    """
    Check if a line is likely a references/bibliography heading based on common keywords.
    """
    low = clean_line(line).lower().strip(" .:-")
    return low in {"references", "reference list", "bibliography", "bibliografie", "literature", "literatuur", "bronnen"}


def is_appendix_heading(line):
    """
    Check if a line is likely an appendix heading based on common keywords.
    """
    low = clean_line(line).lower().strip(" .:-")
    if low in {"appendix", "appendices", "bijlage", "bijlagen"}:
        return True
    if re.match(r"^(appendix|bijlage)\s+[a-z0-9]\b", low):
        return True
    return False


def looks_like_real_paragraph(line):
    """
    Heuristically determine if a line looks like real paragraph text rather than a heading or TOC entry.
    """
    line = clean_line(line)
    if len(line) < 45:
        return False
    if is_toc_entry(line, True):
        return False
    if is_numbered_heading(line) and len(line.split()) <= 12:
        return False
    # A real paragraph usually has verbs/punctuation and several words.
    return len(line.split()) >= 8


def looks_like_real_body_heading(idx, lines):
    """
    Heuristically determine if a line at a given index looks like a real body heading rather than a TOC entry.
    """
    line = clean_line(lines[idx])
    if not line:
        return False
    if is_toc_entry(line, in_toc_region=True):
        return False
    if not (is_numbered_heading(line) or re.match(r"^(introduction|inleiding)\b", line, flags=re.I)):
        return False

    # A real heading is followed soon by paragraph text, not only more TOC rows.
    following = [clean_line(x) for x in lines[idx+1:idx+10] if clean_line(x)]
    para_count = sum(1 for x in following[:8] if looks_like_real_paragraph(x))
    toc_count = sum(1 for x in following[:8] if is_toc_entry(x, True))
    short_heading_count = sum(1 for x in following[:8] if is_numbered_heading(x) and len(x.split()) <= 12)
    return para_count >= 1 and toc_count <= 2 and short_heading_count <= 4

print("Detected language:", detect_language(cleaned_text))


Detected language: en


# Split title/front matter, TOC, main text, references, appendices

In [ ]:
def find_contents_block(lines):
    """
    Search for a block of lines that looks like a Table of Contents, starting with a contents heading and followed by TOC-like entries.
    """
    starts = [i for i, line in enumerate(lines[:300]) if is_contents_heading(line)]
    if not starts:
        return None
    start = starts[0]
    end = start + 1
    # Continue while lines look like TOC rows or compact heading rows.
    blank_streak = 0
    for j in range(start + 1, min(len(lines), start + 450)):
        line = clean_line(lines[j])
        if not line:
            blank_streak += 1
            if blank_streak >= 5:
                break
            continue
        blank_streak = 0
        if is_toc_entry(line, in_toc_region=True) or is_numbered_heading(line) or is_appendix_heading(line):
            end = j + 1
            continue
        # Keep occasional continuation lines in very compact TOCs.
        if len(line.split()) <= 8 and has_trailing_page_number(line):
            end = j + 1
            continue
        # Stop when prose starts.
        if looks_like_real_paragraph(line):
            break
    return (start, end)


def find_body_start(lines):
    """
    Search for the first actual body heading that is followed by real prose, starting after the contents block if it exists.
    """
    toc = find_contents_block(lines)
    search_start = toc[1] if toc else 0

    # Search for the first actual numbered intro/body heading that is followed by real prose.
    for i in range(search_start, min(len(lines), search_start + 700)):
        if looks_like_real_body_heading(i, lines):
            return i

    # Fallback: first real heading after front matter, even if paragraph score is weak.
    for i in range(search_start, min(len(lines), search_start + 700)):
        line = clean_line(lines[i])
        if is_numbered_heading(line) and not is_toc_entry(line, True):
            return i

    # Last fallback: after contents block, else start.
    return search_start


def find_part_boundaries(lines, body_start):
    """
    Search for the start of references and appendices after the body start, using heading cues.
    The first one that appears is considered the main end, and the other defines the next section if it appears later.
    """
    ref_idx = None
    app_idx = None

    for i in range(body_start, len(lines)):
        line = clean_line(lines[i])
        if not line:
            continue
        # Avoid splitting on TOC rows after body_start unless followed by actual appendix/reference content.
        if ref_idx is None and is_reference_heading(line):
            ref_idx = i
            continue
        if app_idx is None and is_appendix_heading(line):
            # Only accept appendix after real body or after references; not compact appendix TOC before actual body.
            if i > body_start + 20:
                app_idx = i
                continue

    first_special = min([x for x in [ref_idx, app_idx] if x is not None], default=len(lines))
    main_end = first_special

    if ref_idx is not None and (app_idx is None or ref_idx < app_idx):
        references_range = (ref_idx, app_idx if app_idx is not None else len(lines))
    else:
        references_range = (None, None)

    if app_idx is not None:
        appendix_range = (app_idx, len(lines))
    else:
        appendix_range = (None, None)

    return main_end, references_range, appendix_range


def join_lines(lines_slice):
    # Preserve headings and paragraphs but remove excessive blank noise.
    out = []
    prev_blank = False
    for line in lines_slice:
        line = clean_line(line)
        if not line:
            if not prev_blank:
                out.append("")
            prev_blank = True
        else:
            out.append(line)
            prev_blank = False
    text = "\n".join(out).strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

contents_block = find_contents_block(lines)
body_start = find_body_start(lines)
main_end, ref_range, app_range = find_part_boundaries(lines, body_start)

titlepage_lines = lines[:body_start]
if contents_block:
    c0, c1 = contents_block
    toc_text = join_lines(lines[c0:c1])
    # Remove TOC from title/front matter.
    titlepage_lines = lines[:c0] + lines[c1:body_start]
else:
    toc_text = ""

titlepage_text = join_lines(titlepage_lines)
main_text = join_lines(lines[body_start:main_end])
references_text = join_lines(lines[ref_range[0]:ref_range[1]]) if ref_range[0] is not None else ""
appendix_text = join_lines(lines[app_range[0]:app_range[1]]) if app_range[0] is not None else ""

# Safety fallback: if main_text is suspiciously tiny while appendix or titlepage contains obvious body headings, recover from full text.
if len(main_text.split()) < 500 and len(cleaned_text.split()) > 3000:
    # Look again for a body start anywhere after front matter that is followed by real prose.
    for i in range(0, len(lines)):
        if looks_like_real_body_heading(i, lines):
            body_start = i
            main_end, ref_range, app_range = find_part_boundaries(lines, body_start)
            main_text = join_lines(lines[body_start:main_end])
            references_text = join_lines(lines[ref_range[0]:ref_range[1]]) if ref_range[0] is not None else ""
            appendix_text = join_lines(lines[app_range[0]:app_range[1]]) if app_range[0] is not None else ""
            break

print("contents_block:", contents_block)
print("body_start:", body_start, lines[body_start] if body_start < len(lines) else None)
print("main words:", len(main_text.split()))
print("references words:", len(references_text.split()))
print("appendix words:", len(appendix_text.split()))
print("MAIN PREVIEW:\n", main_text[:1500])


contents_block: None
body_start: 45 905 1.48 120 1.43 1,025 1.47 41,375 1.55
main words: 3366
references words: 0
appendix words: 0
MAIN PREVIEW:
 905 1.48 120 1.43 1,025 1.47 41,375 1.55
Large
business
(250+
employees)
195 0.32 30 0.36 225 0.32 10,220 0.38
Total 61,250 100 8,410 100 69,660 100 2,669,440 100

Professional, scientific and technical services are the biggest sector in Kent and
Medway's regional economy, with companies engaged in these activities
representing over 17.5% of the total number of enterprises
1. This is followed by the
construction sector at just under 17% and business administration and support
services, making up around 9% of Kent and Medway's economy. 28.5% of all those
in employment who live in Kent are employed in public administration, education and
health, just below the national average of 29.4%
2. The key sectors for Kent and
Medway are reflective of those for the UK. However, Kent, and Medway in particular,
have more construction businesses than the U

# Title page candidates, generic NL/EN

In [ ]:
def extract_date_patterns(text):
    """
    Extract date-like patterns from the text using regular expressions, including various common formats.
    """
    patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTHS})\s+\d{{4}}\b",
        rf"\b(?:{MONTHS})\s+\d{{1,2}}(?:st|nd|rd|th)?[,]?\s+\d{{4}}\b",
        rf"\b(?:{MONTHS})\s+\d{{4}}\b",
    ]
    out = []
    for pat in patterns:
        for m in re.finditer(pat, text or "", flags=re.I):
            val = clean_line(m.group(0))
            if val not in out:
                out.append(val)
    return out

ORG_MARKERS = {
    "university","universiteit","hogeschool","college","school","institute","instituut",
    "hospital","ziekenhuis","medical center","medisch centrum","erasmus","department","faculty","faculteit",
    "province","provincie","municipality","gemeente","ministry","ministerie","company","bedrijf",
    "b.v.","bv","n.v.","nv","inc","ltd","gmbh","foundation","stichting","research","lectoraat",
    "applied sciences", "uas", "mc"
}

DOC_TYPE_MARKERS = {
    "thesis", "graduation thesis", "bsc", "msc", "bachelor", "master", "report", "paper", "portfolio",
    "scriptie", "afstudeer", "afstudeerscriptie", "onderzoeksrapport", "stageverslag"
}

STOP_FRONT_HEADINGS = {"preface", "voorwoord", "abstract", "samenvatting", "contents", "inhoudsopgave", "table of contents"}


def is_org_like(line):
    """
    Check if a line is likely organization-related based on the presence of common keywords.
    """
    low = clean_line(line).lower()
    return any(m in low for m in ORG_MARKERS)


def is_doc_type_line(line):
    """
    Check if a line looks like it indicates the document type (thesis, report, etc.) based on common keywords and length.
    """
    low = clean_line(line).lower()
    return any(m in low for m in DOC_TYPE_MARKERS) and len(line.split()) <= 8


def is_person_like(line):
    """
    Heuristically determine if a line looks like a person name, based on capitalization patterns and length.
    """
    line = clean_line(line)
    if not line or len(line) > 90:
        return False
    if re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", line):
        return False
    if is_org_like(line) or is_doc_type_line(line):
        return False
    line = re.sub(r"\([^)]*\)", "", line).strip()
    words = [w for w in line.split() if re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ]", w)]
    if not 2 <= len(words) <= 5:
        return False
    capitals = 0
    for w in words:
        core = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ'\-]", "", w)
        if core and (core[0].isupper() or core.isupper()):
            capitals += 1
    return capitals >= 2


def extract_titlepage_candidates(titlepage_text):
    """
    Extract candidate lines for title, authors, organizations, document type, and dates from the title page/front matter.
    """
    front_lines = [clean_line(x) for x in titlepage_text.splitlines() if clean_line(x)]
    # Only inspect early front matter before preface/abstract prose.
    usable = []
    for line in front_lines[:50]:
        low = line.lower().strip(" .:-")
        if low in STOP_FRONT_HEADINGS or any(low.startswith(h + " ") for h in STOP_FRONT_HEADINGS):
            break
        usable.append(line)

    dates = extract_date_patterns("\n".join(usable))
    title_candidates = []
    author_candidates = []
    organization_candidates = []
    document_type_candidates = []

    for line in usable[:25]:
        low = line.lower()
        if any(d.lower() in low for d in dates):
            continue
        if is_org_like(line):
            organization_candidates.append(line)
            continue
        if is_doc_type_line(line):
            document_type_candidates.append(line)
            continue
        if is_person_like(line):
            author_candidates.append(re.sub(r"\s*\([^)]*\)", "", line).strip())
            continue
        # Title lines are early, not too short/long, and not pure identifiers.
        if 5 <= len(line) <= 180 and not re.fullmatch(r"[A-Za-z]{0,4}\s*\d{3,12}", line):
            title_candidates.append(line)

    # Deduplicate preserving order.
    def unique(xs, n=8):
        out = []
        seen = set()
        for x in xs:
            key = x.lower()
            if key not in seen:
                seen.add(key)
                out.append(x)
            if len(out) >= n:
                break
        return out

    return {
        "title_candidates": unique(title_candidates, 5),
        "author_candidates": unique(author_candidates, 8),
        "organization_candidates": unique(organization_candidates, 8),
        "document_type_candidates": unique(document_type_candidates, 5),
        "date_candidates": unique(dates, 8),
    }

titlepage_candidates = extract_titlepage_candidates(titlepage_text)
print(json.dumps(titlepage_candidates, ensure_ascii=False, indent=2))


{
  "title_candidates": [
    "Climate Change Risk and Impact Assessment for Kent and Medway Part 2: Industry 1",
    "Part 2:",
    "Climate Change Risk and Impact Assessment for Kent and Medway Part 2: Industry 2",
    "As with the United Kingdom, Kent's economy is predominantly driven by small and",
    "medium enterprises (SMEs). Approximately 90% of companies in Kent and Medway"
  ],
  "author_candidates": [
    "Climate Change Risk",
    "and Impact Assessment",
    "for Kent and Medway",
    "Industry Sector Summary",
    "I. Industry",
    "I.1 Key Characteristics",
    "detailed in Table I-1.",
    "Kent Medway Kent &"
  ],
  "organization_candidates": [],
  "document_type_candidates": [],
  "date_candidates": [
    "November 2019"
  ]
}


# Section detection and chunking

In [ ]:
def normalize_heading(line):
    """
    Normalize a heading line by stripping page numbers, extra whitespace, and common trailing punctuation.
    """
    line = strip_heading_page_number(clean_line(line))
    line = re.sub(r"\s+", " ", line).strip()
    return line


def heading_id(line):
    """
    Generate a stable heading ID based on the line content, using numbered headings or common section names.
    """
    line = normalize_heading(line)
    m = re.match(r"^(?:chapter|hoofdstuk\s+)?(\d+(?:\.\d+){0,6})\b", line, flags=re.I)
    if m:
        return m.group(1)
    low = line.lower()
    for name in ["abstract", "samenvatting", "introduction", "inleiding", "method", "methodology", "methode", "results", "resultaten", "conclusion", "conclusie", "discussion", "discussie"]:
        if low.startswith(name):
            return name
    return None


def is_actual_section_heading(line):
    """
    Heuristically determine if a line looks like an actual section heading in the main body, rather than a TOC entry or front matter heading.
    """
    line = clean_line(line)
    if not line or len(line) > 180:
        return False
    if is_toc_entry(line, True):
        return False
    if is_reference_heading(line) or is_appendix_heading(line):
        return False
    if is_numbered_heading(line):
        # Avoid prose sentences that happen to start with a chapter number.
        stripped = normalize_heading(line)
        return len(stripped.split()) <= 14 and not stripped.endswith((".", "?", "!"))
    low = line.lower().strip(" .:-")
    return low in {"abstract", "samenvatting", "introduction", "inleiding", "method", "methodology", "methode", "results", "resultaten", "conclusion", "conclusie", "discussion", "discussie"}


def detect_sections(main_text):
    """
    Detect sections in the main text based on actual section headings, and return their character offsets and content.
    """
    main_lines = get_lines(main_text)
    headings = []
    char_pos = 0
    for idx, line in enumerate(main_lines):
        if is_actual_section_heading(line):
            headings.append((idx, char_pos, heading_id(line) or f"section_{len(headings)+1}", normalize_heading(line)))
        char_pos += len(line) + 1

    if not headings:
        return [{
            "section_id": "main",
            "heading": "Main text",
            "start_char": 0,
            "end_char": len(main_text),
            "word_count": len(main_text.split()),
            "text": main_text.strip()
        }]

    sections = []
    for h_i, (line_idx, start_char, sid, heading) in enumerate(headings):
        end_char = headings[h_i + 1][1] if h_i + 1 < len(headings) else len(main_text)
        section_text = main_text[start_char:end_char].strip()
        if len(section_text.split()) < 3:
            continue
        sections.append({
            "section_id": sid,
            "heading": heading,
            "start_char": start_char,
            "end_char": end_char,
            "word_count": len(section_text.split()),
            "text": section_text
        })
    return sections


def make_chunk_text(words):
    """
    Join a list of words into a chunk of text, ensuring proper spacing and trimming.
    """
    return " ".join(words).strip()


def chunk_section(section):
    """
    Chunk a section into smaller parts if it exceeds the target word count, with some overlap for context.
    """
    words = section["text"].split()
    if len(words) <= MAX_CHUNK_WORDS:
        if len(words) < MIN_CHUNK_WORDS and section["section_id"] != "main":
            # Keep short sections, but they may be merged later if desired.
            pass
        return [{
            "source_section_id": section["section_id"],
            "source_section_heading": section["heading"],
            "section_part": 1,
            "chunk_type": "main_text",
            "word_count": len(words),
            "text": make_chunk_text(words)
        }]
    chunks = []
    start = 0
    part = 1
    while start < len(words):
        end = min(start + TARGET_CHUNK_WORDS, len(words))
        if end < len(words) and end - start < MIN_CHUNK_WORDS:
            end = min(start + MIN_CHUNK_WORDS, len(words))
        chunks.append({
            "source_section_id": section["section_id"],
            "source_section_heading": section["heading"],
            "section_part": part,
            "chunk_type": "main_text",
            "word_count": end - start,
            "text": make_chunk_text(words[start:end])
        })
        if end >= len(words):
            break
        start = max(0, end - CHUNK_OVERLAP_WORDS)
        part += 1
    return chunks

sections = detect_sections(main_text)
chunks = []
for section in sections:
    chunks.extend(chunk_section(section))

# Give stable chunk IDs.
for i, chunk in enumerate(chunks, 1):
    chunk["chunk_id"] = f"chunk_{i:03d}"

print("Sections:", len(sections))
print("Chunks:", len(chunks))
print("First section preview:", sections[0]["text"][:800] if sections else "")


Sections: 8
Chunks: 9
First section preview: 905 1.48 120 1.43 1,025 1.47 41,375 1.55
Large
business
(250+
employees)


# Save Silver outputs

In [ ]:
detected_language = detect_language(cleaned_text)

silver = {
    "document_id": DOCUMENT_ID,
    "original_file": input_path.name,
    "detected_language": detected_language,
    "processing_layer": "silver",
    "processing_version": "silver_local_llm_v6_generic_body_detection",
    "created_at": datetime.now().isoformat(),
    "statistics": {
        "raw_characters": len(raw_text),
        "cleaned_characters": len(cleaned_text),
        "titlepage_words": len(titlepage_text.split()),
        "toc_words": len(toc_text.split()),
        "main_text_characters": len(main_text),
        "main_text_words": len(main_text.split()),
        "references_words": len(references_text.split()),
        "appendix_words": len(appendix_text.split()),
        "section_count": len(sections),
        "chunk_count": len(chunks),
        "target_chunk_words": TARGET_CHUNK_WORDS,
        "max_chunk_words": MAX_CHUNK_WORDS,
        "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
        "body_start_line_index": body_start,
    },
    "titlepage_candidates": titlepage_candidates,
    "document_parts": {
        "titlepage_text": titlepage_text,
        "table_of_contents": toc_text,
        "main_text": main_text,
        "references_text": references_text,
        "appendix_text": appendix_text,
    },
    "sections": sections,
    "chunks": chunks,
    "local_execution_note": "This Silver layer runs locally and does not call Ollama. It is schema-compatible with later Gold/Gold Meta models.",
    "genericity_note": "No document-specific names, organizations, places, topics or project terms are hardcoded. Body detection uses generic document-structure heuristics for Dutch and English PDFs."
}

silver_json_path = SILVER_DIR / f"{DOCUMENT_ID}_silver.json"
sections_json_path = SILVER_DIR / f"{DOCUMENT_ID}_sections.json"
chunks_jsonl_path = SILVER_DIR / f"{DOCUMENT_ID}_chunks.jsonl"
main_text_path = SILVER_DIR / f"{DOCUMENT_ID}_clean_main_text.txt"

write_json(silver, silver_json_path)
write_json({"document_id": DOCUMENT_ID, "sections": sections}, sections_json_path)
main_text_path.write_text(main_text, encoding="utf-8")
with chunks_jsonl_path.open("w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("Wrote:")
print(silver_json_path)
print(sections_json_path)
print(chunks_jsonl_path)
print(main_text_path)
print("Stats:")
print(json.dumps(silver["statistics"], indent=2))


Wrote:
..\..\Data\silver\doc_01_silver.json
..\..\Data\silver\doc_01_sections.json
..\..\Data\silver\doc_01_chunks.jsonl
..\..\Data\silver\doc_01_clean_main_text.txt
Stats:
{
  "raw_characters": 25204,
  "cleaned_characters": 24263,
  "titlepage_words": 220,
  "toc_words": 0,
  "main_text_characters": 22919,
  "main_text_words": 3366,
  "references_words": 0,
  "appendix_words": 0,
  "section_count": 8,
  "chunk_count": 9,
  "target_chunk_words": 900,
  "max_chunk_words": 1200,
  "chunk_overlap_words": 180,
  "body_start_line_index": 45
}
